# EV Charging Station Recommendation
This notebook explains each implementation step.

### Step 1: Install Required Libraries

In [ ]:
!pip install scikit-fuzzy matplotlib numpy pandas

###Step 2: Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import skfuzzy as fuzz
from skfuzzy import control as ctrl
import matplotlib.pyplot as plt

###Step 3: Define Input and Output Variables

In [ ]:
# Input 1: Distance (km)
distance = ctrl.Antecedent(np.arange(0, 15.1, 0.1), 'distance')

# Input 2: Slot Availability (%)
availability = ctrl.Antecedent(np.arange(0, 101, 1), 'availability')

# Input 3: Charging Cost ($/kWh)
cost = ctrl.Antecedent(np.arange(0, 20.1, 0.1), 'cost')

# Output: Recommendation Score (0-100)
recommendation = ctrl.Consequent(np.arange(0, 101, 1), 'recommendation')

###Step 4: Define Membership Functions

####Distance

In [ ]:
distance['near'] = fuzz.trimf(distance.universe, [0, 0, 6])
distance['moderate'] = fuzz.trimf(distance.universe, [2, 7.5, 13])
distance['far'] = fuzz.trimf(distance.universe, [9, 15, 15])

In [ ]:
distance.view()
plt.show()

####Slot Availability

In [ ]:
availability['low'] = fuzz.trimf(availability.universe, [0, 0, 40])
availability['medium'] = fuzz.trimf(availability.universe, [20, 50, 80])
availability['high'] = fuzz.trimf(availability.universe, [60, 100, 100])

In [ ]:
availability.view()
plt.show()

####Charging Cost

In [ ]:
cost['cheap'] = fuzz.trimf(cost.universe, [0, 0, 8])
cost['moderate'] = fuzz.trimf(cost.universe, [4, 10, 16])
cost['expensive'] = fuzz.trimf(cost.universe, [12, 20, 20])

In [ ]:
cost.view()
plt.show()

####Recommendation Score

In [ ]:
recommendation['low'] = fuzz.trimf(recommendation.universe, [0, 0, 40])
recommendation['medium'] = fuzz.trimf(recommendation.universe, [20, 50, 80])
recommendation['high'] = fuzz.trimf(recommendation.universe, [60, 100, 100])

In [ ]:
recommendation.view()
plt.show()

###Step 6: Create Rule Base

In [ ]:
rule1 = ctrl.Rule(
    distance['near'] & availability['high'] & cost['cheap'],
    recommendation['high']
)

rule2 = ctrl.Rule(
    distance['near'] & availability['medium'] & cost['moderate'],
    recommendation['high']
)

rule3 = ctrl.Rule(
    distance['far'] & availability['low'],
    recommendation['low']
)

rule4 = ctrl.Rule(
    distance['moderate'] & availability['medium'] & cost['moderate'],
    recommendation['medium']
)

rule5 = ctrl.Rule(
    distance['near'] & cost['expensive'],
    recommendation['medium']
)

rule6 = ctrl.Rule(
    availability['low'] & cost['expensive'],
    recommendation['low']
)

rule7 = ctrl.Rule(
    distance['far'] & availability['high'],
    recommendation['medium']
)

rule8 = ctrl.Rule(
    distance['moderate'] & availability['high'] & cost['cheap'],
    recommendation['high']
)

###Step 7: Create Controller

In [ ]:
charging_ctrl = ctrl.ControlSystem([
    rule1,
    rule2,
    rule3,
    rule4,
    rule5,
    rule6,
    rule7,
    rule8
])

system = ctrl.ControlSystemSimulation(charging_ctrl)

###Step 8: Provide Charging Station Values

In [ ]:
system.input['distance'] = 4

system.input['availability'] = 65

system.input['cost'] = 9

###Step 9: Run Fuzzy Inference
The default defuzzification method for a Consequent in scikit-fuzzy is centroid.

In [ ]:
system.compute()

In [ ]:
print(system.output['recommendation'])

###Step 10: Display Result

In [ ]:
print("Recommendation Score:")

print(round(system.output['recommendation'],2))

###Step 11: Show Defuzzification Graph

In [ ]:
recommendation.view(sim=system)

plt.show()

###Step-12 Try Different Test Cases
####30 Test Cases

The fuzzy controller is evaluated using 30 different charging-station conditions. Test Case 4 is the sample scenario used in the project: Distance = 4 km, Slot Availability = 65%, and Charging Cost = 9 $/kWh.

In [ ]:
test_cases = [
    {"distance": 1,  "availability": 95, "cost": 3},
    {"distance": 2,  "availability": 90, "cost": 5},
    {"distance": 3,  "availability": 85, "cost": 6},
    {"distance": 4,  "availability": 65, "cost": 9},   # Project sample
    {"distance": 5,  "availability": 80, "cost": 7},
    {"distance": 6,  "availability": 70, "cost": 8},
    {"distance": 7,  "availability": 60, "cost": 10},
    {"distance": 8,  "availability": 75, "cost": 6},
    {"distance": 9,  "availability": 55, "cost": 11},
    {"distance": 10, "availability": 50, "cost": 10},
    {"distance": 11, "availability": 80, "cost": 5},
    {"distance": 12, "availability": 70, "cost": 7},
    {"distance": 13, "availability": 90, "cost": 8},
    {"distance": 14, "availability": 80, "cost": 10},
    {"distance": 15, "availability": 100, "cost": 4},
    {"distance": 2,  "availability": 30, "cost": 15},
    {"distance": 4,  "availability": 20, "cost": 18},
    {"distance": 6,  "availability": 40, "cost": 12},
    {"distance": 8,  "availability": 35, "cost": 14},
    {"distance": 10, "availability": 25, "cost": 16},
    {"distance": 12, "availability": 20, "cost": 18},
    {"distance": 14, "availability": 10, "cost": 19},
    {"distance": 15, "availability": 5,  "cost": 20},
    {"distance": 9,  "availability": 15, "cost": 17},
    {"distance": 7,  "availability": 10, "cost": 16},
    {"distance": 3,  "availability": 50, "cost": 10},
    {"distance": 5,  "availability": 50, "cost": 12},
    {"distance": 8,  "availability": 65, "cost": 9},
    {"distance": 10, "availability": 85, "cost": 6},
    {"distance": 13, "availability": 75, "cost": 5}
]

results = []

for i, case in enumerate(test_cases, 1):
    test_system = ctrl.ControlSystemSimulation(charging_ctrl)
    test_system.input['distance'] = case['distance']
    test_system.input['availability'] = case['availability']
    test_system.input['cost'] = case['cost']
    test_system.compute()

    score = test_system.output['recommendation']

    results.append({
        "Test Case": i,
        "Distance (km)": case['distance'],
        "Availability (%)": case['availability'],
        "Cost ($/kWh)": case['cost'],
        "Recommendation Score": round(score, 2)
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

###Step-13: Apply different Defuzzification methods and observe the result and compare.
| Method       | Meaning                                          | Typical use                                 |
| ------------ | ------------------------------------------------ | -------------------------------------------- |
| **centroid** | Center of Area / Center of Gravity               | ⭐ Most commonly used                        |
| **bisector** | Divides the aggregated area into two equal areas | Alternative to centroid                     |
| **mom**      | Mean of Maximum                                  | When maximum membership region is important |
| **som**      | Smallest of Maximum                              | Selects the smallest x among maxima          |
| **lom**      | Largest of Maximum                               | Selects the largest x among maxima           |

####1. Centroid
It finds the balance point of the aggregated fuzzy output.

In [ ]:
recommendation.defuzzify_method = 'centroid'
system = ctrl.ControlSystemSimulation(charging_ctrl)
system.input['distance'] = 4
system.input['availability'] = 65
system.input['cost'] = 9
system.compute()
print(round(system.output['recommendation'],2))

In [ ]:
recommendation.view(sim=system)

plt.show()

####2. Bisector

Finds the point that divides the total area under the aggregated membership function into two equal areas.

Think of it as:"Where can I cut the fuzzy output so that the area on both sides is equal?"

In [ ]:
recommendation.defuzzify_method = 'bisector'
system = ctrl.ControlSystemSimulation(charging_ctrl)
system.input['distance'] = 4
system.input['availability'] = 65
system.input['cost'] = 9
system.compute()
print(round(system.output['recommendation'],2))

In [ ]:
recommendation.view(sim=system)

plt.show()

####3. MOM — Mean of Maximum

Finds all points having the maximum membership value and takes their average.

For example: Maximum membership occurs at:

70, 71, 72, 73

MOM = (70+71+72+73)/4
    = 71.5

In [ ]:
recommendation.defuzzify_method = 'mom'
system = ctrl.ControlSystemSimulation(charging_ctrl)
system.input['distance'] = 4
system.input['availability'] = 65
system.input['cost'] = 9
system.compute()
print(round(system.output['recommendation'],2))

In [ ]:
recommendation.view(sim=system)

plt.show()

####4. SOM — Smallest of Maximum

Finds the points having maximum membership and chooses the smallest one.
Maximum points:

70, 71, 72, 73

SOM = 70

In [ ]:
recommendation.defuzzify_method = 'som'
system = ctrl.ControlSystemSimulation(charging_ctrl)
system.input['distance'] = 4
system.input['availability'] = 65
system.input['cost'] = 9
system.compute()
print(round(system.output['recommendation'],2))

In [ ]:
recommendation.view(sim=system)

plt.show()

####5. LOM — Largest of Maximum

Same idea, but chooses the largest maximum point.
Maximum points:

70, 71, 72, 73

LOM = 73

In [ ]:
recommendation.defuzzify_method = 'lom'
system = ctrl.ControlSystemSimulation(charging_ctrl)
system.input['distance'] = 4
system.input['availability'] = 65
system.input['cost'] = 9
system.compute()
print(round(system.output['recommendation'],2))

In [ ]:
recommendation.view(sim=system)

plt.show()

Using For Loop

In [ ]:
methods = ['centroid', 'bisector', 'mom', 'som', 'lom']

for method in methods:
    recommendation.defuzzify_method = method

    system = ctrl.ControlSystemSimulation(charging_ctrl)

    system.input['distance'] = 4
    system.input['availability'] = 65
    system.input['cost'] = 9

    system.compute()

    print(method, system.output['recommendation'])

**Conclusion**: There exist different outputs for different defuzz methods. To identify the best amongst them we need real world data.